## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

from evoc import EVoC
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

np.random.seed(42)

## Loading Text Embeddings

We'll use the 20 Newsgroups dataset, a classic benchmark with documents from different news categories. The real challenge with text clustering is that documents are high-dimensional and noisy—word choice varies even within the same topic. This is exactly where embedding-based approaches shine: modern language models capture semantic meaning regardless of specific word choices.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

print("Loading 20 Newsgroups dataset...\n")

# Select 10 categories for manageable analysis
categories = [
    'alt.atheism', 'comp.graphics', 'comp.sys.ibm.pc.hardware',
    'misc.forsale', 'rec.autos', 'rec.sport.hockey',
    'sci.med', 'sci.space', 'talk.politics.guns', 'talk.religion.misc'
]

newsgroups = fetch_20newsgroups(
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

texts = newsgroups.data
labels_true = newsgroups.target
target_names = newsgroups.target_names

print(f"Loaded {len(texts)} documents from {len(categories)} categories")
print(f"Categories: {target_names}")

## Generating Embeddings

Now we need to convert text to embeddings. We'll use a pre-trained sentence transformer model, which is fast and produces high-quality semantic embeddings. This model was trained on hundreds of millions of text pairs to produce meaningful vector representations of entire documents.

In [ ]:
print("\nGenerating embeddings (this may take a minute)...\n")

# Use a fast, efficient model
model = SentenceTransformer('all-MiniLM-L6-v2')

t0 = time()
embeddings = model.encode(texts, show_progress_bar=True)
time_embed = time() - t0

embeddings = embeddings.astype(np.float32)

print(f"\nEmbeddings generated in {time_embed:.1f} seconds")
print(f"Shape: {embeddings.shape}")
print(f"Type: {embeddings.dtype}")

# Normalize (good practice for semantic embeddings)
scaler = StandardScaler()
X = scaler.fit_transform(embeddings)

print(f"\nNormalized: mean={X.mean():.6f}, std={X.std():.6f}")

## EVōC Clustering

Now we'll run EVōC. Pay attention to the time—we'll compare this with UMAP+HDBSCAN in a moment. But more importantly, we're not telling EVōC anything about the data: no number of clusters, no distance metrics, nothing. It figures out the structure automatically.

In [ ]:
print("Running EVōC clustering...")
t0 = time()
clusterer = EVoC(n_neighbors=15, random_state=42)
labels_evoc = clusterer.fit_predict(X)
time_evoc = time() - t0

n_clusters = len(np.unique(labels_evoc[labels_evoc >= 0]))
n_noise = np.sum(labels_evoc == -1)

print(f"\nEVōC Results (in {time_evoc:.3f}s):")
print(f"  Clusters found: {n_clusters}")
print(f"  Noise points: {n_noise}")
print(f"  Cluster layers: {len(clusterer.cluster_layers_)}")

# Evaluate quality
non_noise = labels_evoc >= 0
if non_noise.sum() > 0:
    ari_evoc = adjusted_rand_score(labels_true[non_noise], labels_evoc[non_noise])
    ami_evoc = adjusted_mutual_info_score(labels_true[non_noise], labels_evoc[non_noise])
    print(f"\nQuality vs true categories:")
    print(f"  ARI: {ari_evoc:.3f}")
    print(f"  AMI: {ami_evoc:.3f}")

## Comparison: UMAP + HDBSCAN

The current "standard approach" for clustering embeddings is to first reduce dimensionality with UMAP, then cluster with HDBSCAN. Let's see how it compares.

This pipeline requires tuning multiple parameters: UMAP's n_components, min_dist, n_neighbors; HDBSCAN's min_cluster_size, etc. We'll use reasonable defaults, but the point is: it's more steps, more decisions to make.

In [ ]:
# Try to import optional dependencies
try:
    import umap
    import hdbscan
    has_comparison = True
except ImportError:
    has_comparison = False
    print("Note: UMAP and/or HDBSCAN not installed.")
    print("Install with: pip install umap-learn hdbscan")

if has_comparison:
    print("\nRunning UMAP + HDBSCAN pipeline...\n")
    
    # UMAP reduction to 10 dimensions (common practice)
    print("  Step 1: UMAP dimensionality reduction...", end=' ', flush=True)
    t0 = time()
    reducer = umap.UMAP(n_components=10, random_state=42, n_jobs=-1)
    X_reduced = reducer.fit_transform(X)
    time_umap = time() - t0
    print(f"Done ({time_umap:.2f}s)")
    
    # HDBSCAN clustering
    print("  Step 2: HDBSCAN clustering...", end=' ', flush=True)
    t0 = time()
    clusterer_hdbscan = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5)
    labels_hdbscan = clusterer_hdbscan.fit_predict(X_reduced)
    time_hdbscan = time() - t0
    print(f"Done ({time_hdbscan:.2f}s)")
    
    time_umap_hdbscan = time_umap + time_hdbscan
    
    n_clusters_umap = len(np.unique(labels_hdbscan[labels_hdbscan >= 0]))
    n_noise_umap = np.sum(labels_hdbscan == -1)
    
    non_noise_umap = labels_hdbscan >= 0
    if non_noise_umap.sum() > 0:
        ari_umap = adjusted_rand_score(labels_true[non_noise_umap], labels_hdbscan[non_noise_umap])
        ami_umap = adjusted_mutual_info_score(labels_true[non_noise_umap], labels_hdbscan[non_noise_umap])
    else:
        ari_umap = ami_umap = 0
    
    print(f"\nUMAP + HDBSCAN Results (in {time_umap_hdbscan:.3f}s):")
    print(f"  Clusters: {n_clusters_umap}")
    print(f"  Noise: {n_noise_umap}")
    print(f"  ARI: {ari_umap:.3f}")
    print(f"  AMI: {ami_umap:.3f}")

## Performance Comparison

Let's put the results side by side. Remember: EVōC is a single-pass algorithm that figures out the structure automatically. The UMAP+HDBSCAN pipeline requires choosing parameters for two separate algorithms.

In [ ]:
print("\n" + "="*75)
print("SUMMARY: EVōC vs UMAP+HDBSCAN")
print("="*75)

print(f"\n{'Metric':<20} | {'EVōC':<18} | {'UMAP+HDBSCAN':<18}")
print("-"*75)
print(f"{'Time':<20} | {time_evoc:<18.3f}s | {time_umap_hdbscan if has_comparison else 'N/A':<18}")
print(f"{'Clusters':<20} | {n_clusters:<18} | {n_clusters_umap if has_comparison else 'N/A':<18}")
print(f"{'Noise points':<20} | {n_noise:<18} | {n_noise_umap if has_comparison else 'N/A':<18}")
print(f"{'ARI':<20} | {ari_evoc:<18.3f} | {ari_umap if has_comparison else 'N/A':<18}")
print(f"{'AMI':<20} | {ami_evoc:<18.3f} | {ami_umap if has_comparison else 'N/A':<18}")
print("="*75)

if has_comparison:
    speedup = time_umap_hdbscan / time_evoc
    print(f"\n✓ EVōC is {speedup:.1f}x faster than UMAP+HDBSCAN")
    print(f"✓ Quality is comparable or better (ARI: {ari_evoc:.3f} vs {ari_umap:.3f})")

print(f"\nKey Insights:")
print(f"  • EVōC found {n_clusters} clusters (true: {len(categories)})")
print(f"  • No parameter tuning needed")
print(f"  • Membership strengths provide confidence scores")
print(f"  • Multiple granularities available via layers")

## Understanding Discovered Clusters

Let's examine what EVōC actually discovered. With semantic embeddings, documents on the same topic should cluster together. Let's look at a few discovered clusters and see what topics they contain.

In [ ]:
# Analyze clusters
print("\nCluster Composition (showing first 3 clusters):\n")
print("="*70)

labels = clusterer.labels_
unique_clusters = sorted([c for c in np.unique(labels) if c >= 0])[:3]

for cluster_id in unique_clusters:
    cluster_mask = labels == cluster_id
    n_docs = cluster_mask.sum()
    
    # What categories are in this cluster?
    true_cats = labels_true[cluster_mask]
    cat_counts = np.bincount(true_cats)
    
    print(f"\nCluster {cluster_id}: {n_docs} documents")
    print(f"  True category distribution:")
    for cat_id, count in enumerate(cat_counts):
        if count > 0:
            pct = count / len(true_cats) * 100
            print(f"    • {target_names[cat_id]:<30} {count:3d} ({pct:5.1f}%)")

## Membership Strength Analysis

One of EVōC's strengths is providing confidence scores. In document clustering, boundary points often represent documents that genuinely could belong to multiple topics. Let's see how confident EVōC is about its assignments.

In [ ]:
strengths = clusterer.membership_strengths_

core_mask = (labels >= 0) & (strengths > 0.8)
boundary_mask = (labels >= 0) & (strengths <= 0.8) & (strengths > 0.5)
uncertain_mask = (labels >= 0) & (strengths <= 0.5)
noise_mask = labels == -1

print("\nDocument Classification by Confidence:")
print(f"\n  High-confidence documents (strength > 0.8):  {core_mask.sum():5d} ({core_mask.sum()/len(labels)*100:5.1f}%)")
print(f"  Boundary documents (0.5 < strength ≤ 0.8):   {boundary_mask.sum():5d} ({boundary_mask.sum()/len(labels)*100:5.1f}%)")
print(f"  Uncertain documents (strength ≤ 0.5):       {uncertain_mask.sum():5d} ({uncertain_mask.sum()/len(labels)*100:5.1f}%)")
print(f"  Outlier documents (not in clusters):        {noise_mask.sum():5d} ({noise_mask.sum()/len(labels)*100:5.1f}%)")

print(f"\nStrength Statistics:")
print(f"  Mean: {strengths.mean():.3f}")
print(f"  Median: {np.median(strengths):.3f}")
print(f"  Std: {strengths.std():.3f}")

print(f"\nWhy this matters:")
print(f"  • High-confidence documents are clearly within their topic")
print(f"  • Boundary documents might be cross-topic or ambiguous")
print(f"  • You can filter by confidence if you need high-quality clusters")

## Visualization

Let's create some visualizations to summarize the results.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Time comparison
if has_comparison:
    methods = ['EVōC', 'UMAP+\nHDBSCAN']
    times = [time_evoc, time_umap_hdbscan]
    axes[0].bar(methods, times, color=['#1f77b4', '#2ca02c'], alpha=0.7)
else:
    axes[0].bar(['EVōC'], [time_evoc], color=['#1f77b4'], alpha=0.7)
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Clustering Speed')
axes[0].grid(axis='y', alpha=0.3)

# 2. Quality
if has_comparison:
    methods = ['EVōC', 'UMAP+HDBSCAN']
    aris = [ari_evoc, ari_umap]
    axes[1].bar(methods, aris, color=['#1f77b4', '#2ca02c'], alpha=0.7)
else:
    axes[1].bar(['EVōC'], [ari_evoc], color=['#1f77b4'], alpha=0.7)
axes[1].set_ylabel('Adjusted Rand Index')
axes[1].set_title('Clustering Quality (ARI)')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)

# 3. Strength distribution
axes[2].hist(strengths[labels >= 0], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[2].axvline(0.8, color='green', linestyle='--', linewidth=2, label='High confidence')
axes[2].axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Boundary')
axes[2].set_xlabel('Membership Strength')
axes[2].set_ylabel('Count')
axes[2].set_title('Confidence Distribution')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

What we learned from clustering text embeddings:

**EVōC's Advantages:**
- Single-pass algorithm (vs. UMAP → HDBSCAN pipeline)
- Automatic parameter discovery
- Comparable or better quality
- Significantly faster
- Provides confidence scores for every assignment

**Practical Takeaways:**
- EVōC works across data modalities (we've seen text, images, synthetic data)
- Semantic embeddings cluster naturally by topic
- The algorithm finds coherent document groups automatically
- You can use membership strengths to filter for high-confidence clusters

**Next Steps:**
- Try with your own document collections
- Explore other embedding models (BERT, RoBERTa, etc.)
- Use membership strengths for quality filtering
- See the benchmark notebook for detailed performance comparisons